## DSPy Ollama Qwen2 Information Extraction Pydantic

#### Load in Python Libraries

In [1]:
import os 
import sys
import re
from dotenv import load_dotenv
load_dotenv()
pythonpath = os.getenv('PYTHONPATH')
if pythonpath:
    sys.path.extend(pythonpath.split(os.pathsep))

import dspy
from transformers import AutoTokenizer, AutoModelForCausalLM
from rich import print
import pandas as pd
import ast

from dspy.teleprompt import BootstrapFewShot, BootstrapFewShotWithRandomSearch
from collections.abc import Iterable


from dspy.evaluate.evaluate import Evaluate

from rouge_score import rouge_scorer
from pydantic import BaseModel

scorer = rouge_scorer.RougeScorer(['rouge1','rouge2', 'rougeL'], use_stemmer=True)
import json

from data.train_examples import train_example_list
from data.valid_examples import dev_example_list
from data.test_example import test_examples_list

/Users/justinvhuang/miniconda3/envs/dspy/lib/python3.11/site-packages/threadpoolctl.py:1214: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


#### Helper Functions

In [2]:
def flatten_list(nested_list):

    for item in nested_list:

        if isinstance(item, Iterable) and not isinstance(item, str):

            yield from flatten_list(item)

        else:

            yield item
            
def validate_ans(example, pred, trace = None):

    gold = re.sub(r'\n|\s+ ', '',dict(example)['info_extracted']).lower()
    print(gold)

    prediction = pred.info_extracted.lower()
    print(prediction)

    scores = scorer.score(gold, prediction)
    score2 = scores['rouge2'][0]
    score1 = scores['rouge1'][0]
    scoreL = scores['rougeL'][0]
    score = (0.2*score1 + 0.3*score2 + 0.5* scoreL)

    print(score)

    return score

def normalize(job_post: str) -> str:
    job_post = job_post.strip('\n')

    job_post = re.sub(r'^[^\w\s]+|[^\w\s]+$', '', job_post, flags=re.UNICODE)

    job_post = job_post.strip('\n')

    return job_post.strip().lower()

#### Load in Data

In [3]:
train_examples=pd.read_csv('/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/Manual Labeling - Sheet1.csv')

dev_examples=pd.read_csv('/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/50examples_for_DSPy_withJson.csv')

test_examples = pd.read_csv("/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/Manual Labeling - Sheet2.csv", header = None)

#### Set Ollama LLM using Qwen 2 from Alibaba

In [4]:
llm = dspy.OllamaLocal(model='qwen2:latest', max_tokens = 4000, temperature=0.0)
dspy.settings.configure(lm=llm)

#### Set Pydantic Class

In [5]:
class JobPostingExtraction(BaseModel):
    position_title: str
    location: str
    work_arrangement: str
    experience: str
    employment_type: str
    pay : str
    degree : str 
    certifications : str 
    required_skills : str

#### Create DSPy Signature 

In [26]:
class InfoExtractor(dspy.Signature):
    """Extract information from a job posting and return the output in a json format if you don't know answer Not Specified. Should be key-value with output as dictionary. """
    job_posting: str = dspy.InputField(desc = "contains job information")
    info_extracted: JobPostingExtraction = dspy.OutputField(desc = "key-value pairs of position_title, location, work_arrangement, experience, employment_type, pay, degree, certifications, required_skills each with less than 20 words all in json format")

#### Crease DSPy Module

In [27]:
class JobPostingModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.info_extraction = dspy.ChainOfThought(InfoExtractor)
    
    def forward(self, job_posting):
        job_posting = job_posting.replace('\n', ' ').replace('“', '"').replace('”', '"')
        job_posting = normalize(job_posting)
        info_extract = self.info_extraction(job_posting=job_posting).info_extracted.replace("```\n", "").replace("```", "")
        # dspy.Suggest(type(json.loads(dict(info_extract)['info_extracted'])) == dict, "Should return a dictionary format")
        # dspy.Suggest(list(json.loads(info_extract).keys()) == ['position_title','location','experience','employment_type','pay','degree','certifications','required_skills'],
        #              "The information extraction needs to have keys position_title','location','experience','employment_type','pay','degree','certifications','required_skills")
        return dspy.Prediction(job_posting= job_posting, info_extracted = info_extract)

In [28]:
uncompiled_module = JobPostingModule()

#### Perform Test Extraction on Data with uncompiled version

In [29]:
print(test_examples[1][5])

EMT-Advanced-Emergency Medical Service
Job Locations
US-TX-Rosenberg
Posted Date
9 months ago
(5/16/2023 11:01 AM)
ID 2023-5265 Job Code DJOB # of Openings 10 Min Start Salary USD $2,008.28/Bi. Category EMS Max Start Salary USD 
$2,421.41/Bi.
Overview

Fort Bend County is ranked as one of the fastest growing counties in the nation. We have capitalized on not only 
the creed of our location, but on the "quality of life" for our families to call home. Our employees are the key to
our success and the heartbeat of our foundation. The diversity and inclusivity of our community is our strength and
at the forefront of a workplace environment welcoming to all. Live Here! Work Here!



Provides emergency medical care to the citizens of Fort Bend County as stated in established standards and 
procedures.



Responsibilities
Provides emergency pre-hospital medical care. 
Completes reports within established timeframes.
Maintains emergency vehicle(s) and inventory of medical supplies.
Signs for and is held accountable for equipment issued and used.
Operates emergency vehicles (i.e. ambulance, squad) per department policy and with due regard to the law.
Responsible for maintaining all current certifications within department guidelines as required.
Certifications shall include EMT-Basic that has successfully completed AEMT and is test eligible and is enrolled in
an EMT Paramedic Program, DSHS EMT-Advanced Certification who is enrolled in an EMT Paramedic Program, and Valid 
State of Texas Driver's License.
Prepares, submits, and maintains clear, concise, and accurate documentation on patient care activities, incident 
reports and other related information as requested.
Assists other employees with their duties.
Performs general housekeeping duties for station and department areas.
Participates in activities and duties related to emergency management during a local state of disaster as directed 
by appropriate county managers.
Qualifications
High School Diploma/GED; Enrolled in College pursing Paramedic Certification and/or EMS Degree.
Certified or Licensed State of Texas EMT-Basic or EMT-Advanced or is eligible to test for EMT-Advanced 
Certification. 
Current Healthcare Provider CPR/AED card.
Pre-hospital experience preferred.
Experience in a high performance ALS system beneficial.
Strong verbal and written communication and organizational skills.
Strong interpersonal skills and ability to deal effectively with the public and other employees. 
Frequent reading, writing, memorization, analyzing, simple math skills, negotiating. Constantly using judgment, 
reasoning, decision-making and teaching.
Ability to complete projects.
Must obtain and maintain a current American Heart Association Advanced Cardiac Life Support certification.
Must complete National Incident Management System (NIMS) 100, 200, 700 and 800 within 90 days of hire.
Must obtain Paramedic Credentials in 24 months after hire date. Subject to emergency call-in and mandatory 
staffing.



SALARY RANGE: EMS Grade EMT-1, $2,008.28 - $2,421.41 biweekly based on qualifications

CLOSING DATE: Upon filling position





Fort Bend County is an equal opportunity employer, committed to non-discrimination in employment on any basis 
including race, color, religion or creed, sex, sexual orientation, gender, gender identity, gender expression, 
pregnancy status (including childbirth and related medical conditions), national origin, ethnicity, citizenship 
status, age (40 and over), physical or mental disability, genetic information, protected military and veteran 
status, political affiliation or beliefs, or any other classification protected by state, federal and local laws, 
unless such classification is a bona fide occupational qualification.

In [30]:
with dspy.context(lm = llm):
    pred = uncompiled_module(job_posting =test_examples[1][5])
    print(pred.info_extracted)

The position being described is for an "Advanced EMT - Emergency Medical Service" role based in Rosenberg, Texas. 
The work arrangement isn't specified but it's likely full-time given the nature of the job.

**Qualifications and Requirements:**
- **Education:** Requires a high school diploma or General Educational Development (GED) certificate along with 
enrollment in college pursuing paramedic certification and/or Emergency Medical Services (EMS) degree.
- **Certifications:** Must be certified or licensed as an EMT-basic or advanced by the state of Texas, possess a 
current healthcare provider CPR/AED card, and obtain American Heart Association Advanced Cardiac Life Support 
(ACLS) certification. Additionally, must complete National Incident Management System (NIMS) training levels 100, 
200, 700, and 800 within the first 90 days of employment.
- **Experience:** Pre-hospital experience is preferred, with beneficial experience in a high performance Advanced 
Life Support (ALS) system. 
- **Skills:** Requires strong verbal and written communication skills, organizational abilities, interpersonal 
skills for effective public interaction, frequent use of reading, writing, memorization, analysis, simple math 
skills, and negotiation capabilities.
- **Project Management:** Must be able to complete projects effectively.

**Closing Date:** The job posting will remain open until the position is filled.

**Employment Policy:** Fort Bend County is committed to equal opportunity employment practices, ensuring 
non-discrimination based on various personal characteristics including race, color, religion, sex, sexual 
orientation, gender identity or expression, pregnancy status, national origin, ethnicity, citizenship status, age 
(40 and over), physical or mental disability, genetic information, military/veteran status, political affiliation, 
beliefs, or any other classification protected by state, federal, or local laws.

This summary encapsulates the key details of the job position including its requirements, skills needed, and 
employment policies.

#### Inspect History and save DSPy program 

In [70]:
#print(llm.inspect_history(n=1))
#uncompiled_module.save("uncompiled_file_pydantic.json")

#### Create Training Examples, Validation(Dev) and Test Examples

In [13]:
print(len(dev_example_list) , len(train_example_list), len(test_examples_list))
train_results = train_example_list
train_contents = list(train_examples['body'])

dev_results = dev_example_list
dev_contents = list(dev_examples.loc[:20,'body'])

test_results = test_examples_list
test_contents = list(test_examples[1])

21 20 10

In [14]:
train_examples_list = [dspy.Example(job_posting=content, info_extracted=result) for content, result in zip(train_contents, train_results)]
dev_examples_list = [dspy.Example(job_posting=content, info_extracted=result) for content, result in zip(dev_contents, dev_results)]
test_examples_list = [dspy.Example(job_posting=content, info_extracted=result) for content, result in zip(test_contents, test_results)]

In [15]:
trainset=train_examples_list
devset=dev_examples_list
testset = test_examples_list

trainset = [x.with_inputs('job_posting') for x in trainset]
devset = [x.with_inputs('job_posting') for x in devset]
testset = [x.with_inputs('job_posting') for x in testset]

#### Test Uncompiled Module on combined Rouge Score

In [21]:
answ = trainset[0]
print(answ)

In [22]:
with dspy.context(lm=llm):
    pred = uncompiled_module(trainset[0].job_posting)
    print(pred.info_extracted)

In [18]:
validate_ans(answ, pred)

{"position_title": "derrickhand","location": "buckhannon, west virginia 26201","work_arrangement": "on-site, 
shifts","experience": "1-2 years of derrickhand experience","employment_type": "full-time","pay": "not 
specified","degree": "high school diploma/ged or equivalent","certification": "cdl b license","required_skills": 
"effective verbal/written communication in english, ability to interact with teams in a fast-paced environment, 
ability to multi-task, basic problem solving, organizational skills, excellent customer-service"}

{ "position_title": "derrickhand", "location": "buckhannon, wv", "work_arrangement": "full-time, day shift", 
"experience": "1-2 years of workover - derrickhand experience required", "employment_type": "rotary drill 
operators, oil and gas", "pay": "$ / hour (not specified)", "degree": "high school diploma, ged, or equivalent 
preferred", "certifications": "cdl b license required to drive rig", "required_skills": [ "effective communication 
(verbal and written)", "team interaction ability", "fast-paced environment handling skills", "basic problem solving
and organizational skills", "excellent customer service skills" ] }

0.6141058867086264

0.6141058867086264

#### BootStrap Few Shot With A Few Examples to Change the Output

In [23]:
teleprompter = BootstrapFewShot(metric=validate_ans) 
compiled = teleprompter.compile(uncompiled_module, trainset=trainset)

#### Test Compiled Version
    * Note: Some reason the compiled version works worst in this scenario, it might be due to the prompt or the metric being chosen that is affecting the output

In [58]:
with dspy.context(lm=llm):
    pred = compiled(job_posting=trainset[0].job_posting)
    print(pred.info_extracted)

Job Title: Derrickhand

Location: Buckhannon, WV

Job Order Number: WV2925359

Post Date: 05/12/2023

**Responsibilities:**
- Perform services on oil and gas wells from elevated positions (rod basket or tubing board).
- Assist with rigging up/down operations.
- Handle tubing, including transferring it between racks and elevators for servicing wells.
- Drive crew truck requiring a CDL B license.

**Skills Required:**
- Rigging coordination skills
- Elevated work handling skills
- Manual lifting techniques
- Fast-paced environment management skills
- Basic problem-solving abilities
- Organizational skills
- Effective communication (verbal and written)
- Team interaction skills

**Additional Information:**
- CDL B license required for driving crew truck.

**Certifications Required:**
- CDL B license

In [59]:
validate_ans(answ, pred)

{"position_title": "derrickhand","location": "buckhannon, west virginia 26201","work_arrangement": "on-site, 
shifts","experience": "1-2 years of derrickhand experience","employment_type": "full-time","pay": "not 
specified","degree": "high school diploma/ged or equivalent","certification": "cdl b license","required_skills": 
"effective verbal/written communication in english, ability to interact with teams in a fast-paced environment, 
ability to multi-task, basic problem solving, organizational skills, excellent customer-service"}

job title: derrickhand

location: buckhannon, wv

job order number: wv2925359

post date: 05/12/2023

**responsibilities:**
- perform services on oil and gas wells from elevated positions (rod basket or tubing board).
- assist with rigging up/down operations.
- handle tubing, including transferring it between racks and elevators for servicing wells.
- drive crew truck requiring a cdl b license.

**skills required:**
- rigging coordination skills
- elevated work handling skills
- manual lifting techniques
- fast-paced environment management skills
- basic problem-solving abilities
- organizational skills
- effective communication (verbal and written)
- team interaction skills

**additional information:**
- cdl b license required for driving crew truck.

**certifications required:**
- cdl b license

0.18966304968589376

0.18966304968589376

In [31]:
compiled.save("compiled_v2_pydantic.json")

#### Do Side by Side Comparieson on Evaluation versus uncompiled vs compiled

In [24]:
evaluation = Evaluate(devset=testset, num_threads=1, display_progress=True, display_table=10,return_outputs=True)

prev_score=evaluation(uncompiled_module, metric=validate_ans)

improved_score=evaluation(compiled, metric=validate_ans)